# RQ3

## Setup

* Follow the instructions on ```Readme.md```
* Inside the *pipeline* folder, run ```python -m src.rq3``` for testing all models OR 
```python -m src.rq3 --model_name``` for a specific model
    * Ex: ```python -m src.rq3 --model_name microsoft/codebert-base```
* For running the default models (without finetuning) add ```-default``` to the model name
    * Ex: ```python -m src.rq3 --model_name microsoft/codebert-base-default```
* Tested models: ```'microsoft/codebert-base', 'Salesforce/codet5-base'```
* Supported languages: ```'python', 'java' , 'cs'```
* Results are saved on ```results/RQ3```

## Results

In [17]:
import pandas as pd
import numpy as np

# Load CSV
csv_path = "../results/RQ3/clone_detection.csv"
df = pd.read_csv(csv_path)

THRESHOLD = 0.5        # set to None to disable filtering
SORT_BY = "f1"         # precision | recall | f1 | mcc | None
DESCENDING = True

if THRESHOLD is not None:
    df = df[np.isclose(df["threshold"], THRESHOLD)]

# df = df[~df["model"].str.contains("-default", na=False)]

#df = df[df["test_dataset"] == "GPTCloneBench"]

if SORT_BY is not None:
    df = df.sort_values(by=SORT_BY, ascending=not DESCENDING)


display(
    df[[
        "model",
        "train_dataset",
        "test_dataset",
        "lan",
        "pairs",
        "threshold",
        "precision",
        "recall",
        "f1",
        "mcc",
        "TP",
        "TN",
        "FP",
        "FN",
    ]].round(4).reset_index(drop=True)
)
df.to_csv("../results/RQ3/best_results.csv", index=False)

,model,train_dataset,test_dataset,lan,pairs,threshold,precision,recall,f1,mcc,TP,TN,FP,FN
0,microsoft/codebert-base,Kamino,GPTCloneBench,csharp,9816,0.5,0.9870,0.9432,0.9646,0.9316,4629,4847,61,279
1,Salesforce/codet5-base,Kamino,GPTCloneBench,csharp,9816,0.5,0.9693,0.9513,0.9602,0.9213,4669,4760,148,239
2,microsoft/codebert-base,Kamino,GPTCloneBench,java,14192,0.5,0.9326,0.9224,0.9274,0.8557,6545,6623,473,551
3,microsoft/codebert-base,Kamino,GPTCloneBench,python,5640,0.5,0.9376,0.9064,0.9217,0.8466,2556,2650,170,264
4,Salesforce/codet5-base,Kamino,GPTCloneBench,java,14192,0.5,0.8965,0.9178,0.9070,0.8121,6513,6344,752,583
5,Salesforce/codet5-base,Kamino,GPTCloneBench,python,5640,0.5,0.8468,0.9113,0.8779,0.7486,2570,2355,465,250
6,Salesforce/codet5-base,Kamino,SemanticCloneBench,python,2000,0.5,0.8685,0.8060,0.8361,0.6858,806,878,122,194
7,Salesforce/codet5-base,Kamino,SemanticCloneBench,csharp,1870,0.5,0.9557,0.7380,0.8328,0.7227,690,903,32,245
8,microsoft/codebert-base,Kamino,SemanticCloneBench,python,2000,0.5,0.9464,0.7410,0.8312,0.7161,741,958,42,259
9,microsoft/codebert-base,Kamino,SemanticCloneBench,csharp,1870,0.5,0.9865,0.7059,0.8229,0.7263,660,926,9,275


In [18]:
import pandas as pd

# Load CSV
df = pd.read_csv(csv_path)

# Detect pretrained vs finetuned
df['type'] = df['model'].apply(
    lambda x: 'Pre-trained' if '-default' in x else 'Fine-tuned'
)

# Simplify model names
def simplify_model(model):
    model = model.lower()

    if "codebert" in model:
        return "CodeBERT"
    elif "codet5" in model:
        return "CodeT5"

    return model

df['model_name'] = df['model'].apply(simplify_model)

# Compact dataset names
dataset_map = {
    "GPTCloneBench": "GCB",
    "SemanticCloneBench": "SCB"
}

df['dataset_short'] = df['test_dataset'].map(dataset_map)

# Pretty language names
lang_map = {
    "python": "Py",
    "java": "Java",
    "csharp": "C\\#",
    "c": "C"
}

df['lang_name'] = df['lan'].map(lang_map)

# Dataset ordering
dataset_order = {
    "GCB": 0,
    "SCB": 1
}

# Sort rows
df = df.sort_values(
    by=['model_name', 'dataset_short', 'pairs'],
    ascending=[True, True, False]
)

# Column definition
col_def = (
    "L{1.7cm}"
    "L{0.9cm}"
    "L{0.7cm}"
    "R{1.0cm}|"
    "R{0.7cm}"
    "R{0.7cm}"
    "R{0.7cm}"
    "R{0.7cm}|"
    "R{0.7cm}"
    "R{0.7cm}"
    "R{0.7cm}"
    "R{0.7cm}"
)

print("\\begin{table*}[t]")
print("\\centering")
print("\\footnotesize")
print("\\addtolength{\\tabcolsep}{-3pt}")
print("\\caption{Results of embedding-based clone detection before and after fine-tuning on the \\textit{Kamino} dataset ($\\theta = 0.5$).}")
print("\\vspace{-4mm}")
print(f"\\begin{{tabular}}{{{col_def}}}")

print("\\toprule")

# Header
print(
    "\\multirow{2}{*}{\\textbf{Model}} & "
    "\\multirow{2}{*}{\\textbf{DS}} & "
    "\\multirow{2}{*}{\\textbf{Lang.}} & "
    "\\multirow{2}{*}{\\textbf{Pairs*}} & "
    "\\multicolumn{4}{c|}{\\textbf{Pre-trained}} & "
    "\\multicolumn{4}{c}{\\textbf{Fine-tuned}} \\\\"
)

print(
    "& & & & "
    "\\textbf{Prec.} & "
    "\\textbf{Rec.} & "
    "\\textbf{F1} & "
    "\\textbf{MCC} & "
    "\\textbf{Prec.} & "
    "\\textbf{Rec.} & "
    "\\textbf{F1} & "
    "\\textbf{MCC} \\\\"
)

print("\\midrule")

# Loop models
for model in df['model_name'].unique():

    df_model = df[df['model_name'] == model]

    unique_rows = (
        df_model[['dataset_short', 'lang_name']]
        .drop_duplicates()
    )

    model_rows = len(unique_rows)

    model_first = True

    datasets = sorted(
        df_model['dataset_short'].unique(),
        key=lambda x: dataset_order.get(x, 999)
    )

    for dataset in datasets:

        df_dataset = df_model[df_model['dataset_short'] == dataset]

        dataset_rows = len(
            df_dataset[['lang_name']]
            .drop_duplicates()
        )

        dataset_first = True

        langs = df_dataset['lang_name'].unique()

        for lang in langs:

            pre_row = df_dataset[
                (df_dataset['lang_name'] == lang) &
                (df_dataset['type'] == 'Pre-trained')
            ].iloc[0]

            fin_row = df_dataset[
                (df_dataset['lang_name'] == lang) &
                (df_dataset['type'] == 'Fine-tuned')
            ].iloc[0]

            model_cell = (
                f"\\multirow{{{model_rows}}}{{*}}{{{model}}}"
                if model_first else ""
            )

            dataset_cell = (
                f"\\multirow{{{dataset_rows}}}{{*}}{{{dataset}}}"
                if dataset_first else ""
            )

            line = (
                f"{model_cell} & "
                f"{dataset_cell} & "
                f"{lang} & "
                f"${int(pre_row['pairs']):,}$ & "
                f"{pre_row['precision']:.2f} & "
                f"{pre_row['recall']:.2f} & "
                f"{pre_row['f1']:.2f} & "
                f"{pre_row['mcc']:.2f} & "
                f"{fin_row['precision']:.2f} & "
                f"{fin_row['recall']:.2f} & "
                f"\\textbf{{{fin_row['f1']:.2f}}} & "
                f"\\textbf{{{fin_row['mcc']:.2f}}} \\\\"
            )

            print(line)

            model_first = False
            dataset_first = False

        print("\\cline{2-12}")
 

print("\\bottomrule")

print(
    "\\multicolumn{12}{c}{"
    "* Balanced positive/negative clone pairs. "
    "GCB = GPTCloneBench; SCB = SemanticCloneBench."
    "} \\\\"
)

print("\\end{tabular}")
print("\\vspace{-5mm}")
print("\\label{tab:rq3Results}")
print("\\end{table*}")

\begin{table*}[t]
\centering
\footnotesize
\addtolength{\tabcolsep}{-3pt}
\caption{Results of embedding-based clone detection before and after fine-tuning on the \textit{Kamino} dataset ($\theta = 0.5$).}
\vspace{-4mm}
\begin{tabular}{L{1.7cm}L{0.9cm}L{0.7cm}R{1.0cm}|R{0.7cm}R{0.7cm}R{0.7cm}R{0.7cm}|R{0.7cm}R{0.7cm}R{0.7cm}R{0.7cm}}
\toprule
\multirow{2}{*}{\textbf{Model}} & \multirow{2}{*}{\textbf{DS}} & \multirow{2}{*}{\textbf{Lang.}} & \multirow{2}{*}{\textbf{Pairs*}} & \multicolumn{4}{c|}{\textbf{Pre-trained}} & \multicolumn{4}{c}{\textbf{Fine-tuned}} \\
& & & & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} & \textbf{MCC} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} & \textbf{MCC} \\
\midrule
\multirow{6}{*}{CodeBERT} & \multirow{3}{*}{GCB} & Java & $14,192$ & 0.50 & 1.00 & 0.67 & 0.01 & 0.93 & 0.92 & \textbf{0.93} & \textbf{0.86} \\
 &  & C\# & $9,816$ & 0.50 & 1.00 & 0.67 & 0.00 & 0.99 & 0.94 & \textbf{0.96} & \textbf{0.93} \\
 &  & Py & $5,640$ & 0.50 & 1.00 & 0.67 & 0.00 & 0.9